# Project 5: Pay Equity Analysis Dashboard

**Research foundation:**
- **Mercer Global Pay Equity Report** — Unadjusted gender pay gap ~18% globally. Adjusted gap (controlling for grade, function, location) narrows to 2–3%, exposing structural placement as the dominant driver.
- **Willis Towers Watson Pay Equity Audit Framework** — Three-stage methodology: job architecture → market pricing → regression analysis. Compa-ratio (actual ÷ midpoint) is the primary equity metric. Target band: 0.85–1.15.
- **WTW Key Finding** — Approximately 66% of total pay inequity is explained by grade distribution differences, not direct pay decisions within the same role.

**Objective:** Replicate a professional pay equity audit using synthetic data calibrated to these benchmarks. Decompose the unadjusted pay gap into structural and direct components using regression.

**Dataset:** 1,000 synthetic employees generated by `pay_equity_generator.py`


## 0. Setup and data loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.family': 'sans-serif',
})

PALETTE = {
    'Male':       '#378ADD',
    'Female':     '#D4537E',
    'Non-binary': '#1D9E75',
}
GENDER_ORDER = ['Male', 'Female', 'Non-binary']
GRADE_ORDER  = ['G1', 'G2', 'G3', 'G4', 'G5', 'G6']

# Load dataset
df = pd.read_csv('pay_equity_data.csv', parse_dates=['hire_date'])

print(f'Loaded {len(df):,} rows x {df.shape[1]} columns')
df.head(3)

## 1. Dataset overview and quality checks

In [ ]:
# Schema and nulls
print('--- dtypes and null counts ---')
print(df.dtypes.to_frame('dtype').join(
    df.isnull().sum().to_frame('nulls')
))

print('\n--- Numeric summary ---')
df[['base_salary', 'compa_ratio', 'range_penetration', 'bonus_pct', 'last_increase_pct']].describe().round(3)

In [ ]:
# Demographic distribution
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col, title in zip(
    axes,
    ['gender', 'ethnicity', 'department'],
    ['Gender', 'Ethnicity', 'Department'],
):
    counts = df[col].value_counts()
    ax.barh(counts.index, counts.values, color='#378ADD', alpha=0.8)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Headcount')

plt.suptitle('Workforce Composition', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 2. Unadjusted pay gap — the headline number

The unadjusted gap is the raw average salary difference between male and female employees with no controls applied. Mercer reports a global average of ~18%. This dataset targets ~14–17%.

In [ ]:
# Average salary by gender
gender_pay = (
    df.groupby('gender')['base_salary']
    .agg(['mean', 'median', 'std', 'count'])
    .round(0)
    .reset_index()
)

male_avg   = gender_pay.loc[gender_pay['gender'] == 'Male',   'mean'].values[0]
female_avg = gender_pay.loc[gender_pay['gender'] == 'Female', 'mean'].values[0]
gap_pct    = (male_avg - female_avg) / male_avg * 100

print(f'Male average salary   : CAD {male_avg:,.0f}')
print(f'Female average salary : CAD {female_avg:,.0f}')
print(f'Unadjusted gap        : {gap_pct:.1f}%  (Mercer global benchmark: ~18%)')

gender_pay

In [ ]:
# Salary distribution by gender — violin plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: violin
ax = axes[0]
gender_data = [df.loc[df['gender'] == g, 'base_salary'] for g in GENDER_ORDER if g in df['gender'].values]
labels = [g for g in GENDER_ORDER if g in df['gender'].values]
colors = [PALETTE[g] for g in labels]

parts = ax.violinplot(gender_data, positions=range(len(labels)), widths=0.7, showmedians=True)
for pc, color in zip(parts['bodies'], colors):
    pc.set_facecolor(color)
    pc.set_alpha(0.6)

ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels)
ax.set_ylabel('Base Salary (CAD)')
ax.set_title('Salary Distribution by Gender', fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

# Right: bar — avg salary by gender
ax2 = axes[1]
bar_data = df.groupby('gender')['base_salary'].mean().reindex(GENDER_ORDER).dropna()
bars = ax2.bar(
    bar_data.index,
    bar_data.values,
    color=[PALETTE[g] for g in bar_data.index],
    width=0.5,
    alpha=0.85,
)
for bar, val in zip(bars, bar_data.values):
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 800,
        f'CAD {val:,.0f}',
        ha='center', va='bottom', fontsize=10,
    )
ax2.set_ylabel('Average Base Salary (CAD)')
ax2.set_title('Average Salary by Gender', fontweight='bold')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax2.set_ylim(0, bar_data.max() * 1.15)

plt.suptitle(f'Unadjusted Gender Pay Gap: {gap_pct:.1f}%', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Grade distribution — the structural driver

WTW's research shows that ~66% of the total pay gap is explained by grade clustering: women and underrepresented minority employees are disproportionately concentrated in lower grades within the same job families. This section makes that pattern visible.

In [ ]:
# Grade distribution by gender
grade_gender = (
    df.groupby(['job_grade', 'gender'])
    .size()
    .reset_index(name='count')
)
grade_gender['pct_within_gender'] = (
    grade_gender.groupby('gender')['count']
    .transform(lambda x: x / x.sum() * 100)
    .round(1)
)

# Pivot for display
pivot = grade_gender.pivot(index='job_grade', columns='gender', values='pct_within_gender')
pivot = pivot.reindex(GRADE_ORDER)
print('Grade distribution (% within each gender group)')
print(pivot.round(1).to_string())

In [ ]:
# Stacked bar: % in each grade by gender — structural clustering visual
fig, ax = plt.subplots(figsize=(11, 5))

grade_pivot = grade_gender.pivot(index='job_grade', columns='gender', values='pct_within_gender').reindex(GRADE_ORDER)

x    = np.arange(len(GRADE_ORDER))
width = 0.25
genders = [g for g in GENDER_ORDER if g in grade_pivot.columns]

for i, gender in enumerate(genders):
    ax.bar(
        x + i * width,
        grade_pivot[gender],
        width=width,
        label=gender,
        color=PALETTE[gender],
        alpha=0.85,
    )

ax.set_xticks(x + width)
ax.set_xticklabels(GRADE_ORDER)
ax.set_ylabel('% of employees within gender group')
ax.set_title(
    'Grade Distribution by Gender — Structural Pay Gap Driver\n'
    '(WTW: ~66% of total pay gap explained by grade clustering)',
    fontweight='bold',
)
ax.legend(title='Gender')
plt.tight_layout()
plt.show()

In [ ]:
# Average salary by grade and gender — shows within-grade gap
grade_salary = (
    df.groupby(['job_grade', 'gender'])['base_salary']
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(11, 5))

grade_pivot_salary = grade_salary.pivot(index='job_grade', columns='gender', values='base_salary').reindex(GRADE_ORDER)

for i, gender in enumerate(genders):
    ax.bar(
        x + i * width,
        grade_pivot_salary[gender] / 1000,
        width=width,
        label=gender,
        color=PALETTE[gender],
        alpha=0.85,
    )

ax.set_xticks(x + width)
ax.set_xticklabels(GRADE_ORDER)
ax.set_ylabel('Average Base Salary (CAD 000s)')
ax.set_title(
    'Average Salary by Grade and Gender\n'
    '(Within-grade gap reflects Mercer adjusted gap of ~2-3%)',
    fontweight='bold',
)
ax.legend(title='Gender')
plt.tight_layout()
plt.show()

## 4. Compa-ratio analysis

The compa-ratio (actual salary ÷ range midpoint) is WTW's primary pay equity metric. A ratio of 1.00 means an employee is paid exactly at the midpoint of their grade band. Target range: 0.85–1.15. Below 0.90 triggers corrective review.

In [ ]:
# Compa-ratio summary by gender
compa_summary = (
    df.groupby('gender')['compa_ratio']
    .agg(['mean', 'median', 'std',
          lambda x: (x < 0.90).sum(),
          lambda x: (x < 0.90).mean() * 100])
    .round(4)
)
compa_summary.columns = ['mean', 'median', 'std', 'n_below_090', 'pct_below_090']
compa_summary = compa_summary.reindex(GENDER_ORDER).dropna()

print('Compa-ratio by gender  (WTW target: 0.85-1.15)')
print(compa_summary.to_string())

In [ ]:
# Compa-ratio distribution — KDE plot
fig, ax = plt.subplots(figsize=(11, 5))

for gender in GENDER_ORDER:
    if gender in df['gender'].values:
        subset = df.loc[df['gender'] == gender, 'compa_ratio']
        subset.plot.kde(ax=ax, label=gender, color=PALETTE[gender], linewidth=2)

# WTW band boundaries
ax.axvline(0.85, color='#E24B4A', linestyle='--', linewidth=1.2, label='Risk threshold (0.85)')
ax.axvline(1.00, color='#444441', linestyle='--', linewidth=1.0, label='Midpoint (1.00)')
ax.axvline(1.15, color='#E24B4A', linestyle='--', linewidth=1.2, label='Upper threshold (1.15)')
ax.axvspan(0.85, 1.15, alpha=0.04, color='green')

ax.set_xlabel('Compa-Ratio')
ax.set_ylabel('Density')
ax.set_title(
    'Compa-Ratio Distribution by Gender\n'
    '(WTW target band: 0.85–1.15 | left shift = underpaid relative to midpoint)',
    fontweight='bold',
)
ax.legend()
ax.set_xlim(0.6, 1.4)
plt.tight_layout()
plt.show()

In [ ]:
# Compa-ratio heatmap by grade and gender
compa_heat = (
    df.groupby(['job_grade', 'gender'])['compa_ratio']
    .mean()
    .unstack()
    .reindex(GRADE_ORDER)
    .round(4)
)

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(
    compa_heat,
    annot=True,
    fmt='.3f',
    cmap='RdYlGn',
    center=1.00,
    vmin=0.80,
    vmax=1.20,
    linewidths=0.5,
    ax=ax,
)
ax.set_title(
    'Average Compa-Ratio by Grade and Gender\n'
    '(Green = above midpoint | Red = below midpoint | Target: 0.85–1.15)',
    fontweight='bold',
)
ax.set_ylabel('Job Grade')
ax.set_xlabel('Gender')
plt.tight_layout()
plt.show()

## 5. Pay band compliance

Employees outside their grade pay band are a pay equity risk. Employees below the band minimum are underpaid and represent the most urgent remediation cases. WTW recommends a corrective review for any cohort where more than 10% of employees are below their range minimum.

In [ ]:
# In-band compliance overall
overall_in_band = df['in_band_flag'].mean() * 100
print(f'Overall in-band compliance: {overall_in_band:.1f}%')

# By gender
print('\nIn-band compliance by gender:')
print(
    df.groupby('gender')['in_band_flag']
    .mean()
    .mul(100)
    .round(1)
    .reindex(GENDER_ORDER)
    .dropna()
    .to_string()
)

# By department
print('\nIn-band compliance by department:')
print(
    df.groupby('department')['in_band_flag']
    .mean()
    .mul(100)
    .round(1)
    .sort_values()
    .to_string()
)

In [ ]:
# Range penetration distribution by gender
fig, ax = plt.subplots(figsize=(11, 4))

for gender in GENDER_ORDER:
    if gender in df['gender'].values:
        subset = df.loc[df['gender'] == gender, 'range_penetration']
        subset.plot.kde(ax=ax, label=gender, color=PALETTE[gender], linewidth=2)

ax.axvline(0.0, color='#E24B4A', linestyle='--', linewidth=1.0, label='Range minimum')
ax.axvline(0.5, color='#444441', linestyle='--', linewidth=1.0, label='Midpoint (0.5)')
ax.axvline(1.0, color='#E24B4A', linestyle='--', linewidth=1.0, label='Range maximum')

ax.set_xlabel('Range Penetration  (0 = min, 0.5 = mid, 1 = max)')
ax.set_ylabel('Density')
ax.set_title(
    'Pay Range Penetration by Gender\n'
    '(Left shift = employees positioned low in their band)',
    fontweight='bold',
)
ax.legend()
ax.set_xlim(-0.2, 1.3)
plt.tight_layout()
plt.show()

## 6. Regression — adjusted pay gap decomposition

This OLS regression replicates the core methodology of a formal pay equity audit. By adding controls progressively, we can isolate the portion of the pay gap attributable to structural grade placement versus direct within-grade pay decisions.

**Model sequence:**
1. Baseline — gender only (unadjusted gap)
2. Structural controls — add job grade
3. Full audit model — add department, performance, and tenure

In [ ]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Encode variables
reg_df = df.copy()
reg_df['is_female']    = (reg_df['gender'] == 'Female').astype(int)
reg_df['is_nonbinary'] = (reg_df['gender'] == 'Non-binary').astype(int)
reg_df['is_urm']       = reg_df['ethnicity'].isin([
    'Black or African American', 'Hispanic or Latino'
]).astype(int)
reg_df['log_salary']   = np.log(reg_df['base_salary'])

print('Regression variables ready.')
print(f"is_female mean     : {reg_df['is_female'].mean():.3f}")
print(f"is_urm mean        : {reg_df['is_urm'].mean():.3f}")

In [ ]:
# Model 1: Unadjusted — gender only
m1 = ols(
    'log_salary ~ is_female + is_nonbinary',
    data=reg_df,
).fit()

female_coef_m1 = m1.params['is_female']
gap_pct_m1     = (np.exp(female_coef_m1) - 1) * 100

print('MODEL 1 — Unadjusted (gender only)')
print(f'Female coefficient : {female_coef_m1:.4f}')
print(f'Implied pay gap    : {gap_pct_m1:.1f}%')
print(f'R-squared          : {m1.rsquared:.4f}')
print(m1.summary().tables[1])

In [ ]:
# Model 2: Structural control — add job grade
m2 = ols(
    'log_salary ~ is_female + is_nonbinary + C(job_grade)',
    data=reg_df,
).fit()

female_coef_m2 = m2.params['is_female']
gap_pct_m2     = (np.exp(female_coef_m2) - 1) * 100

print('MODEL 2 — Structural control (+ job grade)')
print(f'Female coefficient : {female_coef_m2:.4f}')
print(f'Implied pay gap    : {gap_pct_m2:.1f}%')
print(f'R-squared          : {m2.rsquared:.4f}')

In [ ]:
# Model 3: Full audit model — grade + department + performance + tenure
m3 = ols(
    'log_salary ~ is_female + is_nonbinary + is_urm '
    '+ C(job_grade) + C(department) '
    '+ performance_rating + years_tenure',
    data=reg_df,
).fit()

female_coef_m3 = m3.params['is_female']
gap_pct_m3     = (np.exp(female_coef_m3) - 1) * 100
urm_coef_m3    = m3.params['is_urm']
urm_gap_pct    = (np.exp(urm_coef_m3) - 1) * 100

print('MODEL 3 — Full audit model (grade + dept + perf + tenure)')
print(f'Female coefficient : {female_coef_m3:.4f}  → adjusted gap: {gap_pct_m3:.1f}%')
print(f'URM coefficient    : {urm_coef_m3:.4f}  → adjusted gap: {urm_gap_pct:.1f}%')
print(f'R-squared          : {m3.rsquared:.4f}')
print(m3.summary().tables[1])

In [ ]:
# Gap decomposition waterfall — visualise structural vs. direct gap
total_gap      = abs(gap_pct_m1)
residual_after_grade = abs(gap_pct_m2)
residual_full  = abs(gap_pct_m3)
structural     = total_gap - residual_after_grade
dept_perf      = residual_after_grade - residual_full
direct         = residual_full

labels  = ['Total (unadjusted)', 'Structural\n(grade)', 'Dept/Perf/Tenure', 'Direct pay\n(adjusted)']
values  = [total_gap, structural, dept_perf, direct]
colors  = ['#378ADD', '#D4537E', '#EF9F27', '#1D9E75']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(labels, values, color=colors, width=0.5, alpha=0.88)
for bar, val in zip(bars, values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.2,
        f'{val:.1f}%',
        ha='center', va='bottom', fontsize=11, fontweight='bold',
    )

ax.set_ylabel('Pay Gap (%)')
ax.set_title(
    'Pay Gap Decomposition — Structural vs. Direct Component\n'
    '(WTW benchmark: structural component drives ~66% of total gap)',
    fontweight='bold',
)
ax.set_ylim(0, total_gap * 1.25)

structural_pct = structural / total_gap * 100
ax.text(
    0.98, 0.95,
    f'Structural share: {structural_pct:.0f}% of total gap',
    transform=ax.transAxes,
    ha='right', va='top',
    fontsize=10,
    color='#444441',
)

plt.tight_layout()
plt.show()

print(f'\nSummary:')
print(f'  Total unadjusted gap : {total_gap:.1f}%')
print(f'  Structural component : {structural:.1f}% ({structural_pct:.0f}% of total)')
print(f'  Dept/Perf/Tenure     : {dept_perf:.1f}%')
print(f'  Direct (adjusted)    : {direct:.1f}%  (Mercer benchmark: 2-3%)')

## 7. Promotion rate analysis

Promotion inequity is a leading indicator of future pay gaps. Mercer research shows that a sustained 10 percentage point promotion gap at G3/G4 compounds into a 12–15% salary difference over a 5-year period.

In [ ]:
# Promotion rate by gender and grade
promo = (
    df.groupby(['job_grade', 'gender'])['promoted_last_2yr']
    .mean()
    .mul(100)
    .reset_index()
)
promo.columns = ['job_grade', 'gender', 'promotion_rate_pct']

promo_pivot = promo.pivot(index='job_grade', columns='gender', values='promotion_rate_pct').reindex(GRADE_ORDER)

fig, ax = plt.subplots(figsize=(11, 5))
for i, gender in enumerate(genders):
    ax.bar(
        x + i * width,
        promo_pivot[gender],
        width=width,
        label=gender,
        color=PALETTE[gender],
        alpha=0.85,
    )

ax.set_xticks(x + width)
ax.set_xticklabels(GRADE_ORDER)
ax.set_ylabel('Promotion Rate (%)')
ax.set_title(
    'Promotion Rate by Grade and Gender\n'
    '(Mercer: 10pp gap at G3/G4 compounds into 12-15% pay gap over 5 years)',
    fontweight='bold',
)
ax.legend(title='Gender')
plt.tight_layout()
plt.show()

## 8. Merit increase analysis — controlling for performance

In [ ]:
# Merit increase by gender and performance rating
merit = (
    df.groupby(['performance_rating', 'gender'])['last_increase_pct']
    .mean()
    .reset_index()
)

merit_pivot = merit.pivot(
    index='performance_rating', columns='gender', values='last_increase_pct'
)

print('Average merit increase % by gender and performance rating')
print('(If women receive lower increases at the same rating, the process carries a bias signal)')
print(merit_pivot.round(2).to_string())

fig, ax = plt.subplots(figsize=(10, 4))
x2 = np.arange(5)
for i, gender in enumerate(genders):
    if gender in merit_pivot.columns:
        ax.plot(
            x2, merit_pivot[gender],
            marker='o', label=gender, color=PALETTE[gender], linewidth=2,
        )

ax.set_xticks(x2)
ax.set_xticklabels([1, 2, 3, 4, 5])
ax.set_xlabel('Performance Rating')
ax.set_ylabel('Average Merit Increase (%)')
ax.set_title(
    'Merit Increase by Performance Rating and Gender\n'
    '(Controlling for performance isolates process-level bias)',
    fontweight='bold',
)
ax.legend(title='Gender')
plt.tight_layout()
plt.show()

## 9. Key findings summary

In [ ]:
print('=' * 65)
print('PAY EQUITY AUDIT — KEY FINDINGS SUMMARY')
print('=' * 65)

print(f"""
1. UNADJUSTED GENDER PAY GAP:  {gap_pct:.1f}%
   Mercer global benchmark: ~18%
   Women earn {100 + gap_pct_m1:.1f} cents for every dollar earned by men.

2. ADJUSTED GAP (full model):  {abs(gap_pct_m3):.1f}%
   Mercer benchmark: 2-3% after controlling for grade, dept, performance.
   After controlling for structural factors, a small but statistically
   significant direct pay gap remains in the data.

3. STRUCTURAL COMPONENT:       {structural:.1f}% of total gap
   WTW benchmark: ~66% of gap explained by grade clustering.
   Women are overrepresented in G1-G3 and underrepresented in G5-G6.
   This is the primary remediation target: hiring and promotion equity.

4. COMPA-RATIO GAP:            {compa_summary.loc['Male', 'mean'] - compa_summary.loc['Female', 'mean']:.4f}
   Male avg compa-ratio:   {compa_summary.loc['Male', 'mean']:.4f}
   Female avg compa-ratio: {compa_summary.loc['Female', 'mean']:.4f}
   WTW target band: 0.85-1.15 for all groups.

5. PAY BAND COMPLIANCE:        {overall_in_band:.1f}% in-band overall
   Departments with lowest compliance are highest priority for
   corrective salary review.

RECOMMENDED ACTIONS (Mercer / WTW framework):
  - Audit hiring and promotion criteria at G3/G4 for structural bias
  - Set compa-ratio targets for female and URM employees in next cycle
  - Flag all employees below 0.90 compa-ratio for manager review
  - Establish pay equity monitoring as a quarterly HR metric
""")
print('=' * 65)